# 03. Outsource and Retrieve (Sec. IV-D, IV-E, Fig. 6)

This notebook shows searchable encryption end to end. Alice keeps seven documents on three servers,
each filed under one keyword. The servers store only ciphertexts. To search, Alice derives the search
key $dsk$ for a keyword from her password-derived $K_u$ and sends it to every server. Each server tests
every item it holds against $dsk$ and returns the matches, which only Alice can decrypt.

In [1]:
import dataclasses
import time

from ris_sem_project import protocol
from ris_sem_project.display import SCAN_HEADERS, payload_size, peek, scan_row, table
from ris_sem_project.index import ScanOutcome
from ris_sem_project.params import PARAM_I, setup
from ris_sem_project.parties.qpase_user import QPASEUser
from ris_sem_project.primitives.encoding import bytes_int, unpack
from ris_sem_project.primitives.ring import Ring
from ris_sem_project.primitives.symmetric import aes_key, decrypt

PASSWORD = "correct horse battery staple"


def flip(data):
    middle = len(data) // 2
    return data[:middle] + bytes([data[middle] ^ 1]) + data[middle + 1 :]

## Setup, Register and Login (recap of notebook 02)

$N = 3$ servers, threshold $t = 2$. Alice registers and logs in with $\{S_1, S_2\}$. After L3 her
session holds $K_u$, $(pk_u, sk_u)$ and the AES-256 key for documents, all derived from $K_u$.

In [2]:
pp = setup(PARAM_I, N=3, t=2, mu=5)
ring = Ring(pp.q, pp.n)
servers = protocol.setup_servers(pp)
alice = QPASEUser(pp, "alice", PASSWORD)
registered = protocol.register(alice, servers)
session = protocol.login(alice, servers, (1, 2))
print(f"Register on S_1, S_2, S_3: pk_u = {peek(registered.key_message.user_pk)}")
print(f"Login with S_1, S_2: K_u = {peek(session.k_u)}, same as Register: {session.k_u == registered.k_u}")

Register on S_1, S_2, S_3: pk_u = fa4c…4625 (1312 B)
Login with S_1, S_2: K_u = e2c1…e938 (1792 B), same as Register: True


## The documents

Each document gets an item id $i$, one keyword $w$ and the server it goes to. Alice may pick any
$S_i$ ("arbitrary $S_i$" in O1). Three documents share the keyword `medical` and two share `travel`.

In [3]:
CORPUS = [
    (1, "medical", 1, "Blood test results, Riverside Lab, 14 Sep 2026. Haemoglobin 14.1 g/dL, LDL cholesterol 2.9 mmol/L, HbA1c 5.2%. All within range, no follow-up needed."),
    (2, "invoice", 2, "Invoice 2026-0417, Northside Plumbing. Replaced kitchen tap and fixed the leak under the sink: 2.5 h labour plus parts, EUR 186.40, due 30 Oct 2026."),
    (3, "medical", 3, "Physiotherapy referral from Dr. Okafor. Lumbar MRI shows a mild disc bulge at L4-L5 with no nerve compression. Six sessions, twice a week."),
    (4, "travel", 1, "Flight booking KX42QP: Lisbon LIS to Berlin BER on 3 Nov 2026, departs 09:40, seat 23A, one checked bag."),
    (5, "meeting", 2, "Project sync, 21 Oct. Priya ships the export fix on Friday. Tom drafts the Q4 budget. Next review 4 Nov in room 3.12."),
    (6, "medical", 2, "Vaccination record: influenza 2026/27 dose given 2 Oct 2026, batch FL2611, left arm. Tetanus booster due 2031."),
    (7, "travel", 3, "Hotel Spreeblick confirmation 88231: 3-7 Nov 2026, 4 nights, double room, breakfast included, check-in from 15:00."),
]
DOCUMENTS = {item_id: text.encode() for item_id, _, _, text in CORPUS}
print(table(("item i", "keyword w", "server", "|d|", "document d (start)"), [
    (item_id, keyword, f"S_{server}", f"{len(DOCUMENTS[item_id])} B", text[:48] + "...")
    for item_id, keyword, server, text in CORPUS
]))

item i  keyword w  server  |d|    document d (start)
------  ---------  ------  -----  ---------------------------------------------------
1       medical    S_1     149 B  Blood test results, Riverside Lab, 14 Sep 2026. ...
2       invoice    S_2     148 B  Invoice 2026-0417, Northside Plumbing. Replaced ...
3       medical    S_3     138 B  Physiotherapy referral from Dr. Okafor. Lumbar M...
4       travel     S_1     104 B  Flight booking KX42QP: Lisbon LIS to Berlin BER ...
5       meeting    S_2     117 B  Project sync, 21 Oct. Priya ships the export fix...
6       medical    S_2     110 B  Vaccination record: influenza 2026/27 dose given...
7       travel     S_3     114 B  Hotel Spreeblick confirmation 88231: 3-7 Nov 202...


## O1 for item 1, step by step (Sec. IV-D)

U computes $dsk = HKDF(K_u, w)$, samples $\rho \leftarrow \mathbb{Z}_q$ and sets $v = PRF(dsk, \rho)$. Then
$C = Enc(K_u, d)$, $sig_C = Sign(sk_u, (\rho, v, C))$, $Ct = Enc(dsk, (\rho, v, sig_C))$ and
$sig_{Ct} = Sign(sk_u, (Ct, C, i))$. U sends $(Ct, C, i, sig_{Ct})$ to the chosen $S_i$. The values
$\rho$, $v$ and $sig_C$ travel only inside $Ct$. The cell opens $Ct$ with Alice's own $dsk$ to print them.

In [4]:
item_id, keyword, server_id, _ = CORPUS[0]
dsk = alice.search_key(keyword)
request = alice.outsource(keyword, DOCUMENTS[item_id], item_id)
item = request.item
rho, v, sig_c = unpack(decrypt(aes_key(dsk), item.index_ct), 3)
print(f"w = {keyword!r}, dsk = HKDF(K_u, w): {peek(dsk)}")
print(f"rho:    {bytes_int(rho)} (in Z_q, {len(rho)} B)")
print(f"v:      {peek(v)}")
print(f"C:      {peek(item.data_ct)}  (12 B nonce + |d| = {len(DOCUMENTS[item_id])} B + 16 B tag)")
print(f"sig_C:  {peek(sig_c)}")
print(f"Ct:     {peek(item.index_ct)}")
print(f"sig_Ct: {peek(request.signature)}")
print(f"OutsourceRequest to S_{server_id}: Ct {len(item.index_ct)} B + C {len(item.data_ct)} B + sig_Ct {len(request.signature)} B = {payload_size(request)} B with the user id")

w = 'medical', dsk = HKDF(K_u, w): dad3…84a2 (1792 B)
rho:    251556844 (in Z_q, 4 B)
v:      387a…4ba6 (1792 B)
C:      530e…ec31 (177 B)  (12 B nonce + |d| = 149 B + 16 B tag)
sig_C:  fdad…2e3b (2420 B)
Ct:     a7b1…4530 (4256 B)
sig_Ct: 1576…2a3a (2420 B)
OutsourceRequest to S_1: Ct 4256 B + C 177 B + sig_Ct 2420 B = 6858 B with the user id


## O2 at $S_1$

$S_1$ stores $(Ct, C, i)$ only if $Ver(pk_u, (Ct, C, i), sig_{Ct})$ holds. The cell first sends a copy
with one bit of $C$ flipped in transit, which $S_1$ rejects, then the real request.

In [5]:
server = servers[server_id]
forged = dataclasses.replace(request, item=dataclasses.replace(item, data_ct=flip(item.data_ct)))
forged_accepted = server.outsource_store(forged)
accepted = server.outsource_store(request)
print(f"S_{server_id}: Ver(pk_u, (Ct, C', i), sig_Ct) with C' tampered: {forged_accepted}")
print(f"S_{server_id}: Ver(pk_u, (Ct, C, i), sig_Ct): {accepted}")
print(f"S_{server_id} stores for 'alice': {[(s.item_id, len(s.index_ct), len(s.data_ct)) for s in server.storage['alice']]} as (i, |Ct|, |C|)")

S_1: Ver(pk_u, (Ct, C', i), sig_Ct) with C' tampered: False
S_1: Ver(pk_u, (Ct, C, i), sig_Ct): True
S_1 stores for 'alice': [(1, 4256, 177)] as (i, |Ct|, |C|)


## Outsource the other documents

The same O1 and O2 for items 2 to 7, each to its own server, through `protocol.outsource`.

In [6]:
outsourced = {item_id: request}
for item_id, keyword, server_id, _ in CORPUS[1:]:
    run = protocol.outsource(alice, servers[server_id], keyword, DOCUMENTS[item_id], item_id)
    outsourced[item_id] = run.request
print(table(("item i", "keyword w", "server", "|Ct|", "|C|", "|sig_Ct|"), [
    (item_id, keyword, f"S_{server_id}", len(outsourced[item_id].item.index_ct), len(outsourced[item_id].item.data_ct), len(outsourced[item_id].signature))
    for item_id, keyword, server_id, _ in CORPUS
]))

item i  keyword w  server  |Ct|  |C|  |sig_Ct|
------  ---------  ------  ----  ---  --------
1       medical    S_1     4256  177  2420
2       invoice    S_2     4256  176  2420
3       medical    S_3     4256  166  2420
4       travel     S_1     4256  132  2420
5       meeting    S_2     4256  145  2420
6       medical    S_2     4256  138  2420
7       travel     S_3     4256  142  2420


## What each server holds

A server keeps $(ID_u, pk_u, \sigma, \mu_u)$ and opaque $(Ct, C, i)$ triples. The keyword is not stored
anywhere, and $Ct$ has the same size for every item.

In [7]:
for i, server in servers.items():
    record = server.users["alice"]
    print(f"S_{i}: pk_u {peek(record.user_pk)}, sigma {peek(record.hint)}, mu_u = {record.fail_count}")
    print(table(("item i", "|Ct|", "|C|", "Ct"), [
        (stored.item_id, len(stored.index_ct), len(stored.data_ct), peek(stored.index_ct)) for stored in server.storage["alice"]
    ]))
    print()

S_1: pk_u fa4c…4625 (1312 B), sigma a540…71ab (1792 B), mu_u = 1
item i  |Ct|  |C|  Ct
------  ----  ---  ------------------
1       4256  177  a7b1…4530 (4256 B)
4       4256  132  b801…100f (4256 B)

S_2: pk_u fa4c…4625 (1312 B), sigma a540…71ab (1792 B), mu_u = 1
item i  |Ct|  |C|  Ct
------  ----  ---  ------------------
2       4256  176  d6c0…c96d (4256 B)
5       4256  145  ce80…716f (4256 B)
6       4256  138  174e…4f84 (4256 B)

S_3: pk_u fa4c…4625 (1312 B), sigma a540…71ab (1792 B), mu_u = 0
item i  |Ct|  |C|  Ct
------  ----  ---  ------------------
3       4256  166  c0e5…40d4 (4256 B)
7       4256  142  bfe6…be11 (4256 B)



## The servers cannot read the documents

$C$ is encrypted under a key derived from $K_u$, and $Ct$ under $dsk$. No server ever sees either key.
The cell searches each stored $C$ and $Ct$ for a 12-byte piece of every document and tries to decrypt
with every 32-byte value the server holds: its key share $k_i$, $pk_i$, $pk_u$ and $\sigma$.

In [8]:
leak_rows = []
for i, server in servers.items():
    record = server.users["alice"]
    held = {
        "k_i": ring.to_bytes(server.key.k_i),
        "pk_i": ring.to_bytes(server.key.pk_i),
        "pk_u": record.user_pk,
        "sigma": record.hint,
    }
    for stored in server.storage["alice"]:
        d = DOCUMENTS[stored.item_id]
        pieces = [d[j : j + 12] for j in range(0, len(d) - 12, 12)]
        found = sum(piece in stored.data_ct or piece in stored.index_ct for piece in pieces)
        opened = [name for name, value in held.items() for blob in (stored.data_ct, stored.index_ct) if decrypt(aes_key(value), blob) is not None]
        leak_rows.append((i, stored.item_id, found, len(pieces), opened))
print(table(
    ("server", "item i", "plaintext pieces found in C or Ct", "held values that open C or Ct"),
    [(f"S_{i}", item_id, f"{found}/{total}", ", ".join(opened) or "none") for i, item_id, found, total, opened in leak_rows],
))

server  item i  plaintext pieces found in C or Ct  held values that open C or Ct
------  ------  ---------------------------------  -----------------------------
S_1     1       0/12                               none
S_1     4       0/8                                none
S_2     2       0/12                               none
S_2     5       0/9                                none
S_2     6       0/9                                none
S_3     3       0/11                               none
S_3     7       0/9                                none


## R1: Alice searches for `medical` (Sec. IV-E)

U computes $dsk = HKDF(K_u, w)$ and $sig_{dsk} = Sign(sk_u, dsk)$ and sends both to all $N$ servers.
Every server must be asked, because Alice did not have to remember where she put each document.
Receiving $dsk$ lets each server test its items for this one keyword. It still cannot open $C$.

In [9]:
QUERY = "medical"
expected_ids = [item_id for item_id, keyword, _, _ in CORPUS if keyword == QUERY]
request = alice.retrieve_request([QUERY])
print(f"w = {QUERY!r}")
print(f"dsk = HKDF(K_u, w):  {peek(request.search_keys[0])}")
print(f"sig_dsk:             {peek(request.signature)}")
for i in servers:
    print(f"U -> S_{i}: RetrieveRequest, {payload_size(request)} B")

w = 'medical'
dsk = HKDF(K_u, w):  dad3…84a2 (1792 B)
sig_dsk:             8ace…333f (2420 B)
U -> S_1: RetrieveRequest, 4217 B
U -> S_2: RetrieveRequest, 4217 B
U -> S_3: RetrieveRequest, 4217 B


## R2 on every server

$S_i$ checks $Ver(pk_u, dsk, sig_{dsk})$ and then, for each stored $(Ct, C, i)$:
$(\rho, v, sig_C) = Dec(dsk, Ct)$, $Ver(pk_u, (\rho, v, C), sig_C)$, and $v = PRF(dsk, \rho)$. An item that
passes all three goes into $L_i$. A `-` means the step did not run because an earlier one failed.
Items filed under another keyword fail at $Dec$, since their $Ct$ was encrypted under a different $dsk$.

In [10]:
start = time.perf_counter()
scans = {i: server.retrieve_scan(request) for i, server in servers.items()}
r2_seconds = time.perf_counter() - start
for i, scan in scans.items():
    print(f"S_{i}")
    print(table(SCAN_HEADERS, [scan_row(item_scan) for item_scan in scan.items]))
    print(f"L_{i} = {[s.item_id for s in scan.response.results]}")
    print()
print(f"R2 on all {len(servers)} servers: {r2_seconds:.2f} s")

S_1
item  Dec(dsk, Ct)  rho        v (from Ct)         Ver sig_C  PRF(dsk, rho)       v == PRF  outcome
----  ------------  ---------  ------------------  ---------  ------------------  --------  --------------
1     ok            251556844  387a…4ba6 (1792 B)  True       387a…4ba6 (1792 B)  True      MATCH
4     fail          -          -                   -          -                   -         DECRYPT_FAILED
L_1 = [1]

S_2
item  Dec(dsk, Ct)  rho        v (from Ct)         Ver sig_C  PRF(dsk, rho)       v == PRF  outcome
----  ------------  ---------  ------------------  ---------  ------------------  --------  --------------
2     fail          -          -                   -          -                   -         DECRYPT_FAILED
5     fail          -          -                   -          -                   -         DECRYPT_FAILED
6     ok            100931954  1774…1d5f (1792 B)  True       1774…1d5f (1792 B)  True      MATCH
L_2 = [6]

S_3
item  Dec(dsk, Ct)  rho        v (f

## R3 at Alice

U merges $L_1, L_2, L_3$, runs the same check on each returned item with her own $pk_u$ and $dsk$
(she does not trust the servers' answer), and decrypts $d = Dec(K_u, C)$ for every item that passes.

In [11]:
opened = alice.retrieve_open([QUERY], [scan.response for scan in scans.values()])
print(table(("from", *SCAN_HEADERS), [
    (", ".join(f"S_{i}" for i in o.server_ids), *scan_row(o.scan)) for o in opened
]))
print()
for o in opened:
    print(f"item {o.item_id} from {', '.join(f'S_{i}' for i in o.server_ids)}, d = Dec(K_u, C):")
    print(f"  {o.data.decode()}")
recovered = {o.item_id: o.data for o in opened if o.data is not None}

from  item  Dec(dsk, Ct)  rho        v (from Ct)         Ver sig_C  PRF(dsk, rho)       v == PRF  outcome
----  ----  ------------  ---------  ------------------  ---------  ------------------  --------  -------
S_1   1     ok            251556844  387a…4ba6 (1792 B)  True       387a…4ba6 (1792 B)  True      MATCH
S_3   3     ok            144980847  f1a3…b8ce (1792 B)  True       f1a3…b8ce (1792 B)  True      MATCH
S_2   6     ok            100931954  1774…1d5f (1792 B)  True       1774…1d5f (1792 B)  True      MATCH

item 1 from S_1, d = Dec(K_u, C):
  Blood test results, Riverside Lab, 14 Sep 2026. Haemoglobin 14.1 g/dL, LDL cholesterol 2.9 mmol/L, HbA1c 5.2%. All within range, no follow-up needed.
item 3 from S_3, d = Dec(K_u, C):
  Physiotherapy referral from Dr. Okafor. Lumbar MRI shows a mild disc bulge at L4-L5 with no nerve compression. Six sessions, twice a week.
item 6 from S_2, d = Dec(K_u, C):
  Vaccination record: influenza 2026/27 dose given 2 Oct 2026, batch FL2611, lef

In [12]:
assert sorted(recovered) == expected_ids, "exactly the items filed under the keyword are returned"
assert all(recovered[i] == DOCUMENTS[i] for i in expected_ids), "each recovered d equals the original"
assert all(o.scan.outcome is ScanOutcome.MATCH for o in opened), "U's own re-check passes for every returned item"
print(f"{QUERY!r}: recovered items {sorted(recovered)}, equal to the originals")

'medical': recovered items [1, 3, 6], equal to the originals


## A keyword with no documents

Alice searches for `tax`. Every stored $Ct$ was encrypted under some other $dsk$, so every row stops at
$Dec$, and every $L_i$ is empty.

In [13]:
empty = protocol.retrieve(alice, servers, "tax")
for i, scan in empty.scans.items():
    print(f"S_{i}")
    print(table(SCAN_HEADERS, [scan_row(item_scan) for item_scan in scan.items]))
    print(f"L_{i} = {[s.item_id for s in scan.response.results]}")
    print()
print(f"documents returned for 'tax': {empty.documents}")

S_1
item  Dec(dsk, Ct)  rho  v (from Ct)  Ver sig_C  PRF(dsk, rho)  v == PRF  outcome
----  ------------  ---  -----------  ---------  -------------  --------  --------------
1     fail          -    -            -          -              -         DECRYPT_FAILED
4     fail          -    -            -          -              -         DECRYPT_FAILED
L_1 = []

S_2
item  Dec(dsk, Ct)  rho  v (from Ct)  Ver sig_C  PRF(dsk, rho)  v == PRF  outcome
----  ------------  ---  -----------  ---------  -------------  --------  --------------
2     fail          -    -            -          -              -         DECRYPT_FAILED
5     fail          -    -            -          -              -         DECRYPT_FAILED
6     fail          -    -            -          -              -         DECRYPT_FAILED
L_2 = []

S_3
item  Dec(dsk, Ct)  rho  v (from Ct)  Ver sig_C  PRF(dsk, rho)  v == PRF  outcome
----  ------------  ---  -----------  ---------  -------------  --------  --------------
3     fail

## A server that alters what it stores

$S_3$ flips one bit of item 3's $Ct$. AES-GCM authenticates $Ct$, so $Dec(dsk, Ct)$ fails. $S_2$ swaps
item 6's $C$ for item 5's. $Ct$ still opens under $dsk$, but $sig_C$ was made over the original $C$,
so $Ver(pk_u, (\rho, v, C), sig_C)$ fails. Neither item is returned. Item 1 on $S_1$ is untouched and
comes back as before.

In [14]:
originals = {i: list(server.storage["alice"]) for i, server in servers.items()}
item_5_c = next(s.data_ct for s in servers[2].storage["alice"] if s.item_id == 5)
servers[3].storage["alice"] = [dataclasses.replace(s, index_ct=flip(s.index_ct)) if s.item_id == 3 else s for s in originals[3]]
servers[2].storage["alice"] = [dataclasses.replace(s, data_ct=item_5_c) if s.item_id == 6 else s for s in originals[2]]

tampered = protocol.retrieve(alice, servers, QUERY)
for i in (2, 3):
    print(f"S_{i}")
    print(table(SCAN_HEADERS, [scan_row(item_scan) for item_scan in tampered.scans[i].items]))
    print()
tampered_outcomes = {s.item_id: s.outcome for i in (2, 3) for s in tampered.scans[i].items if s.item_id in (3, 6)}
print(f"outcomes: item 3 {tampered_outcomes[3].name}, item 6 {tampered_outcomes[6].name}")
print(f"returned for {QUERY!r}: items {[o.item_id for o in tampered.opened]}")

for i, items in originals.items():
    servers[i].storage["alice"] = items

S_2
item  Dec(dsk, Ct)  rho        v (from Ct)         Ver sig_C  PRF(dsk, rho)  v == PRF  outcome
----  ------------  ---------  ------------------  ---------  -------------  --------  -----------------
2     fail          -          -                   -          -              -         DECRYPT_FAILED
5     fail          -          -                   -          -              -         DECRYPT_FAILED
6     ok            100931954  1774…1d5f (1792 B)  False      -              -         SIGNATURE_INVALID

S_3
item  Dec(dsk, Ct)  rho  v (from Ct)  Ver sig_C  PRF(dsk, rho)  v == PRF  outcome
----  ------------  ---  -----------  ---------  -------------  --------  --------------
3     fail          -    -            -          -              -         DECRYPT_FAILED
7     fail          -    -            -          -              -         DECRYPT_FAILED

outcomes: item 3 DECRYPT_FAILED, item 6 SIGNATURE_INVALID
returned for 'medical': items [1]


## Summary of every query

In [15]:
travel = protocol.retrieve(alice, servers, "travel")
queries = [
    (QUERY, sorted(recovered), {i: [s.item_id for s in scan.response.results] for i, scan in scans.items()}),
    ("tax", [o.item_id for o in empty.opened], {i: [s.item_id for s in scan.response.results] for i, scan in empty.scans.items()}),
    (QUERY + " (S_2, S_3 tampered)", [o.item_id for o in tampered.opened if o.data is not None], {i: [s.item_id for s in scan.response.results] for i, scan in tampered.scans.items()}),
    ("travel", [o.item_id for o in travel.opened if o.data is not None], {i: [s.item_id for s in scan.response.results] for i, scan in travel.scans.items()}),
]
for keyword, matched, lists in queries:
    returned_by = ", ".join(f"S_{i}: {ids}" for i, ids in lists.items())
    print(f"{keyword!r}: matched items {matched}; returned by {returned_by}")

'medical': matched items [1, 3, 6]; returned by S_1: [1], S_2: [6], S_3: [3]
'tax': matched items []; returned by S_1: [], S_2: [], S_3: []
'medical (S_2, S_3 tampered)': matched items [1]; returned by S_1: [1], S_2: [], S_3: []
'travel': matched items [4, 7]; returned by S_1: [4], S_2: [], S_3: [7]


## Checks

In [16]:
assert sorted(recovered) == expected_ids and all(recovered[i] == DOCUMENTS[i] for i in expected_ids), "medical returns its three documents"
assert not forged_accepted and accepted, "O2 accepts only a request whose sig_Ct verifies"
assert all(found == 0 and not opened for _, _, found, _, opened in leak_rows), "no server finds plaintext or holds a key for C or Ct"
assert empty.documents == [] and all(
    s.outcome is ScanOutcome.DECRYPT_FAILED for scan in empty.scans.values() for s in scan.items
), "a keyword with no documents fails Dec on every item and returns nothing"
assert tampered_outcomes == {3: ScanOutcome.DECRYPT_FAILED, 6: ScanOutcome.SIGNATURE_INVALID}, "tampered items fail the check"
assert [o.item_id for o in tampered.opened] == [1] and tampered.documents == [DOCUMENTS[1]], "only the untouched item is returned"
assert travel.documents == [DOCUMENTS[4], DOCUMENTS[7]], "travel returns items 4 and 7"
print("all outsource and retrieve checks passed")

all outsource and retrieve checks passed
